In [1]:
import os
import sys

# # Standard interactive replacement for the 'parent directory' hack
# parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
# sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from PPRCalculator import PPRCalculator
from utils import species_groups

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [2]:
# models:
# 116 - Northern Gulf Od Saint Lawrence #1
# 462 - Northern Gulf Od Saint Lawrence #2
# 227 - Iceland 1950
# 711 - Danajon Bank (2010)
# 432 - Ningaloo (2007)
# 180 - Bamboung (2006)
# 725 - Guinea (2004)

# model_numbers = [432, 180, 711, 725, 116, 227]
df = pd.DataFrame(species_groups)
model_numbers = sorted(list(df['model_number'].unique()))
collect_mc_sppr = True
n_samples = 100

# run over models and save SPPRs for different approaches:

In [3]:
model_number = 227
model = PPRCalculator(model_number)

In [4]:
# sppr, _, _ = model.sample_SPPR_new(DET_modeling='as_PP_part', DET_TE_vals=1)
# sppr, _ = model.sample_SPPR_new_forced_balance()
# sppr

In [5]:
path = 'real_models/'
saved_models = [int(f.split('[')[1].split(']')[0]) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

# saved_models = []
# model_numbers = [227, 462]

for model_number in tqdm(model_numbers):
    print(model_number)
    # print(model_number)
    if model_number in saved_models:
        continue
    # create model calculator:
    try:
        model = PPRCalculator(model_number)

        # create name for the model
        year = model.get_model().model_year
        name = model.get_model().model_name
        model_name = f'[{model_number}] {name} {year}'
        # print(model_name)
        # don't treat models with more than one DET row yet:
        if len(model.get_DET_seq()) > 1:
            # print(f'    model {model_name} failed - has more than 1 DET row')
            # print('-'*20)
            continue
        # don't treat models where catch is 0:
        if model.catch.sum() == 0:
            # print(f'    model {model_name} has 0 catch :(')
            # print('-'*20)
            # continue
            pass

        # collect simple spprs:
        # print(f'running {model_name}...')
        results = dict()
        spacial_balance = dict()
        results['SPPR_1986'] = model.SPPR_1986()
        results['SPPR_1995_mTL_global_TE_01'] = model.SPPR_1995(global_TE=0.1)  # This is the 1995 model
        results['SPPR_1995_TL2_global_TE_01'] = model.SPPR_1995_TL_fix(global_TE=0.1) # should be overestimation for omnivorous species
        results['SPPR_1995_mTL_global_mTE'] = model.SPPR_1995(global_TE='mean')
        results['SPPR_1995_TL2_global_mTE'] = model.SPPR_1995_TL_fix(global_TE='mean') # should be overestimation for omnivorous species
        results['SPPR_1995_TL_Jensened_global_mTE'], _, _ = model.SPPR_EwE_Ulanowicz(TE_option='global', global_TE='mean', use_EE=False)  # by Jensen: should be between SPPR_1995_mTL_global_TE and SPPR_1995_TL2_global_TE
        results['SPPR_1995_TL_TE_Jensened'], _, _ = model.SPPR_EwE_Ulanowicz(TE_option='TE', global_TE=None, use_EE=False)  # by Jensen: should be between SPPR_1995_mTL_global_TE and SPPR_1995_TL2_global_TE
        results['SPPR_EwE'], _, _ = model.SPPR_EwE(TE_option='TE', use_EE=True, return_paths=False, silent=True)  # This is the EwE model
        results['SPPR_2015'], _, _ = model.SPPR_2015(only_pp_det=True)  # this is 2015's model
        results['SPPR_new_2015'], _, _ = model.SPPR_new(TE_option='TE', DET_TE_vals=1)  # should be like SPPR_2015
        results['SPPR_new_full'], _, _ = model.SPPR_new(TE_option='With Egestion', DET_TE_vals=1)  # TE includes egestion
        results['SPPR_new_GE'], _, _  = model.SPPR_new(TE_option='GE', DET_TE_vals=1)  # egestion counts as respiration
        _, results['SPPR_symbolic_GE_import_as_PP'], _, _  = model.SPPR_symbolic(TE_option='GE', diet_import_option='as_PP')
        _, results['SPPR_symbolic_GE_import_as_DC'], e, v  = model.SPPR_symbolic(TE_option='GE', diet_import_option='as_DC')
        spacial_balance['SPPR_symbolic_GE_import_as_DC'] = model.is_sppr_balanced(results['SPPR_symbolic_GE_import_as_DC'], diet_import_equations=(e, v))
        _, results['SPPR_symbolic_TE_import_as_PP'], _, _  = model.SPPR_symbolic(TE_option='TE', diet_import_option='as_PP')
        _, results['SPPR_symbolic_TE_import_as_DC'], e, v  = model.SPPR_symbolic(TE_option='TE', diet_import_option='as_DC')
        spacial_balance['SPPR_symbolic_TE_import_as_DC'] = model.is_sppr_balanced(results['SPPR_symbolic_TE_import_as_DC'], diet_import_equations=(e, v))
        _, results['SPPR_symbolic_full_import_as_PP'], _, _  = model.SPPR_symbolic(TE_option='With Egestion', diet_import_option='as_PP')
        _, results['SPPR_symbolic_full_import_as_DC'], e, v  = model.SPPR_symbolic(TE_option='With Egestion', diet_import_option='as_DC')
        spacial_balance['SPPR_symbolic_full_import_as_DC'] = model.is_sppr_balanced(results['SPPR_symbolic_full_import_as_DC'], diet_import_equations=(e, v))

        if collect_mc_sppr:
            # results[f'SPPR_mc_GE_{5}_{10}'], _ = model.monte_carlo_SPPR(
            #     n_samples=n_samples, TE_error_percent=5, TE_error_cut_percent=10, kind='symbolic', TE_option='GE', silent=True
            # )
            results[f'SPPR_mc_GE_{10}_{20}'], _, _, _, _ = model.monte_carlo_SPPR(
                n_samples=n_samples, TE_error_percent=10, TE_error_cut_percent=20, kind='new', TE_option='GE', silent=True
            )
            # results[f'SPPR_mc_full_{10}_{20}'], _, _, _, _ = model.monte_carlo_SPPR(
            #     n_samples=n_samples, TE_error_percent=10, TE_error_cut_percent=20, kind='new', TE_option='With Egestion', silent=True
            # )
            # results[f'SPPR_mc_GE_{20}_{50}'], _ = model.monte_carlo_SPPR(
            #     n_samples=n_samples, TE_error_percent=20, TE_error_cut_percent=50, kind='symbolic', TE_option='GE', silent=True
            # )

        sanity_checks = dict()
        for method_name, sppr in results.items():
            if method_name not in spacial_balance.keys():
                is_sppr_balanced, inflow, outflow = model.is_sppr_balanced(sppr)
            else:
                is_sppr_balanced, inflow, outflow = spacial_balance[method_name]
            is_model_balanced, p, q = model.is_model_balanced()
            sanity_checks[method_name] = {
                'model is balanced': is_model_balanced,
                'SPPR did not explode': bool(np.all(sppr >= -1e-10)),
                'SPPR is balanced': is_sppr_balanced,
                'inflow': inflow,
                'outflow': outflow,
                'DC rows sum to 1': bool(np.all(np.isclose(model._DC.sum(axis=1)[model.get_Regular_seq()], 1, atol=1e-5, rtol=1e-5))),
                'catch is not negative': all(model.catch >= 0)
            }

        # save results into groups_df:
        groups_df = model.get_groups_df()
        index = groups_df.index
        groups_df['ge'] = model.GE
        groups_df['catch'] = model.catch
        groups_df['p'] = model.p
        groups_df['q'] = model.q
        groups_df['predation'] = model.predation
        groups_df['M0'] = model.M0
        groups_df['egestion'] = model.egestion
        groups_df['biomass_accum'] = model.growth
        for k, v in results.items():
            v = v.reindex(index, fill_value=1)
            if isinstance(v, pd.Series):
                v = v.to_frame()
            if len(v.columns) > 1:
                v_orig = v.copy()
                v['sum all'] = v_orig.sum(axis=1)
                v['sum PP'] = v_orig[model.get_PP_seq()].sum(axis=1)
                v['sum inner'] = v_orig.drop(columns=model.get_Import_seq()).sum(axis=1)
                v = v.rename(columns=model.seq2name)
                for col in v.columns:
                    groups_df[k + ' (' + str(col) + ')'] = v[col]
            else:
                groups_df[k + ' (sum all)'] = v.iloc[:, 0]
                groups_df[k + ' (sum inner)'] = v.iloc[:, 0]

        # save:
        with pd.ExcelWriter(f'real_models/{model_name}.xlsx') as writer:
            groups_df.to_excel(writer, sheet_name='sppr')
            pd.DataFrame(sanity_checks).to_excel(writer, sheet_name='sanity_checks')
        # print('-'*20)
        saved_models.append(model_number)

    except:
        # print(f'    model {model_number} failed for unknown reason')
        # print('-'*20)
        continue

failed_models = [i for i in model_numbers if i not in saved_models]
print(f'failed on models ({len(failed_models)}/{len(model_numbers)}): {failed_models}')
print(f'success on models ({len(saved_models)}/{len(model_numbers)}): {saved_models}')

  0%|          | 0/168 [00:00<?, ?it/s]

2
7
24
28
29
34
40
41
46
48
53
58
63
64
68
99
105
107
108
111
112
115
116
118
125
133
135
136
137
145
153
168
172
175
179
180
183
189
217
218
221
227
232
239
240
241
242
243
246
247
252
266
267
269
279
282
291
298
305
307
311
312
318
320
323
324
325
328
335
400
401
403
405
406
407
410
411
412
413
414
415
417
429
431
432
433
435
438
439
441
443
444
446
447
448
450
452
456
457
459
461
462
464
465
466
467
468
473
477
478
479
485
486
487
488
489
490
495
496
497
499
500
501
502
503
504
505
506
513
518
519
520
521
526
537
567
568
608
633
634
637
646
650
651
654
655
658
663
664
669
674
675
677
680
682
687
689
691
692
693
703
704
705
706
707
711
725
726
failed on models (49/168): [np.int64(24), np.int64(34), np.int64(48), np.int64(53), np.int64(58), np.int64(63), np.int64(64), np.int64(99), np.int64(133), np.int64(175), np.int64(179), np.int64(183), np.int64(239), np.int64(252), np.int64(279), np.int64(307), np.int64(328), np.int64(335), np.int64(400), np.int64(401), np.int64(403), np.int64(40

# compare PPRs of different SPPRs:

In [8]:
sppr_col_names = [
    'SPPR_1986',
    'SPPR_1995_mTL_global_TE_01',
    'SPPR_1995_TL2_global_TE_01',
    'SPPR_1995_mTL_global_mTE',
    'SPPR_1995_TL2_global_mTE',
    'SPPR_1995_TL_Jensened_global_mTE',
    'SPPR_1995_TL_TE_Jensened',
    'SPPR_EwE',
    'SPPR_2015',
    'SPPR_new_2015',
    'SPPR_new_full',
    'SPPR_new_GE',
    'SPPR_symbolic_GE_import_as_PP',
    'SPPR_symbolic_GE_import_as_DC',
    'SPPR_symbolic_TE_import_as_PP',
    'SPPR_symbolic_TE_import_as_DC',
    # 'SPPR_symbolic_full_import_as_PP',
    # 'SPPR_symbolic_full_import_as_DC',
    # 'SPPR_mc_TE_5_10',
    'SPPR_mc_GE_10_20',
    # 'SPPR_mc_full_10_20',
    # 'SPPR_mc_TE_20_50',
]
path = 'real_models/'
saved_models = [int(f.split('[')[1].split(']')[0]) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
saved_models.sort()

In [ ]:
# sppr_col_names = list(results.keys())
df_PPR_all = pd.DataFrame(index=sppr_col_names)
df_PPR_inner = pd.DataFrame(index=sppr_col_names)
df_sppr_explode = pd.DataFrame(index=sppr_col_names)
df_sppr_balanced = pd.DataFrame(index=sppr_col_names)
df_outflow_minus_inflow = pd.DataFrame(index=sppr_col_names)

for model_number in saved_models:
    # read model:
    try:
        model = PPRCalculator(model_number)
    except:
        print(f'    model {model_number} failed for unknown reason')
        print('-'*20)
        continue
    
    year = model.get_model().model_year
    name = model.get_model().model_name
    filename = f'[{model_number}] {name} {year}'
    directory = 'real_models'

    # don't treat models where catch is 0:
    if model.catch.sum() == 0:
        print(f'    model {filename} has 0 catch :(')
        print('-'*20)
        continue

    # get sppr_sum columns:
    df_all = pd.read_excel(f'{directory}/{filename}.xlsx', sheet_name='sppr')
    df_all = df_all[['group_seq', 'group_name', 'trophic_info', 'tl', 'ge', 'ee', 'catch', 'q', 'p', 'biomass'] + [c for c in df_all.columns if 'sum all' in c]]
    df_all = df_all.set_index('group_seq').rename(columns={
        c: c.split(' ')[0] for c in df_all.columns
    })
    df_all.columns = df_all.columns.str.replace('sppr', 'SPPR')

    df_inner = pd.read_excel(f'{directory}/{filename}.xlsx', sheet_name='sppr')
    df_inner = df_inner[['group_seq', 'group_name', 'trophic_info', 'tl', 'ge', 'ee', 'catch', 'q', 'p', 'biomass'] + [c for c in df_inner.columns if 'sum inner' in c]]
    df_inner = df_inner.set_index('group_seq').rename(columns={
        c: c.split(' ')[0] for c in df_inner.columns
    })
    df_inner.columns = df_inner.columns.str.replace('sppr', 'SPPR')

    # calculate PPRs:
    df_checks = pd.read_excel(f'{directory}/{filename}.xlsx', sheet_name='sanity_checks', index_col=0)
    c = sppr_col_names[0]
    df_sppr_explode.loc['model is fine', f'{filename}'] = all(df_checks.loc[['model is balanced', 'DC rows sum to 1', 'catch is not negative'], c])
    df_sppr_balanced.loc['model is fine', f'{filename}'] = all(df_checks.loc[['model is balanced', 'DC rows sum to 1', 'catch is not negative'], c])
    df_PPR_all.loc['model is balanced', f'{filename}'] = all(df_checks.loc[['model is balanced'], c])
    df_PPR_all.loc['DC rows sum to 1', f'{filename}'] = all(df_checks.loc[['DC rows sum to 1'], c])
    df_PPR_all.loc['catch is not negative', f'{filename}'] = all(df_checks.loc[['catch is not negative'], c])
    df_PPR_all.loc['model is fine', f'{filename}'] = all(df_checks.loc[['model is balanced', 'DC rows sum to 1', 'catch is not negative'], c])
    df_PPR_inner.loc['model is balanced', f'{filename}'] = all(df_checks.loc[['model is balanced'], c])
    df_PPR_inner.loc['DC rows sum to 1', f'{filename}'] = all(df_checks.loc[['DC rows sum to 1'], c])
    df_PPR_inner.loc['catch is not negative', f'{filename}'] = all(df_checks.loc[['catch is not negative'], c])
    df_PPR_inner.loc['model is fine', f'{filename}'] = all(df_checks.loc[['model is balanced', 'DC rows sum to 1', 'catch is not negative'], c])
    for c in sppr_col_names:
        c = c.replace('sppr', 'SPPR')
        df_PPR_all.loc[c, f'{filename}'] = model.get_PPR(df_all[c], only_inner=False)
        df_PPR_inner.loc[c, f'{filename}'] = model.get_PPR(df_inner[c], only_inner=True)
        df_sppr_explode.loc[c, f'{filename}'] = all(df_checks.loc[['SPPR did not explode'], c])
        df_sppr_balanced.loc[c, f'{filename}'] = all(df_checks.loc[['SPPR is balanced'], c])
        df_outflow_minus_inflow.loc[c, f'{filename}'] = df_checks.loc['outflow', c] - df_checks.loc['inflow', c]

with pd.ExcelWriter(f'../output/PPRs.xlsx') as writer:
    df_PPR_all.T.to_excel(writer, sheet_name='PPRs all')
    df_PPR_inner.T.to_excel(writer, sheet_name='PPRs inner')
    df_sppr_explode.T.to_excel(writer, sheet_name='SPPR did not explode')
    df_sppr_balanced.T.to_excel(writer, sheet_name='SPPR is balanced')
    df_outflow_minus_inflow.T.to_excel(writer, sheet_name='outflow minus inflow')

    model [125] Ria Formosa 1996 has 0 catch :(
--------------------
    model [180] Bamboung 2006 has 0 catch :(
--------------------
    model [221] Garonne 1990 has 0 catch :(
--------------------
    model [269] Looe Key National Marine Sanctuary 1980 has 0 catch :(
--------------------
    model [298] Paraná River Floodplain 1992 has 0 catch :(
--------------------
    model [318] Tamiahua 1989 has 0 catch :(
--------------------
    model [324] Virgin Islands 1960 has 0 catch :(
--------------------
    model [452] Miramare 2000 has 0 catch :(
--------------------
    model [500] North Benguela 1600 has 0 catch :(
--------------------
    model [504] South Benguela 1600 has 0 catch :(
--------------------
    model [567] Arachania 1992 has 0 catch :(
--------------------
    model [568] Barra Del Chuy 1992 has 0 catch :(
--------------------


In [10]:
df_sppr_explode.loc['SPPR_new_2015', '[411] Falkland Islands 1990']

False